# 15 — Upskilling Recommendation Engine
V1: if/else rules. V2: sentence-transformer + cosine similarity.

In [ ]:

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

PROC = r'../data/processed'

emp_gap = pd.read_csv(f'{PROC}/employee_skill_gap_summary.csv')
skill_gap_detailed = pd.read_csv(f'{PROC}/skill_gap_detailed.csv')
org_gap = pd.read_csv(f'{PROC}/org_skill_gap.csv')
sw = pd.read_csv(f'{PROC}/software_skills_processed.csv')
mapping_df = pd.read_csv(f'{PROC}/role_mapping.csv')
print(f"Loaded data. Employees: {len(emp_gap)}")


In [ ]:

# ══════════════════════════════════════════════
# V1 — Rule-Based Recommendation Engine
# ══════════════════════════════════════════════
# Rules:
# 1. HIGH severity org gap + employee missing → "Critical Priority" recommendation
# 2. MEDIUM severity + employee missing → "Growth Priority" recommendation
# 3. Employee's top 3 most important gaps → personalized skill order

high_severity_skills = set(org_gap[org_gap['severity'] == 'HIGH']['skill'])
medium_severity_skills = set(org_gap[org_gap['severity'] == 'MEDIUM']['skill'])

def v1_recommend(employee_id, skill_gap_detailed, high_skills, medium_skills, top_n=5):
    emp_gaps = skill_gap_detailed[
        (skill_gap_detailed['employee_id'] == employee_id) & 
        (skill_gap_detailed['is_gap'] == True)
    ].sort_values('importance', ascending=False)
    
    recommendations = []
    for _, row in emp_gaps.iterrows():
        skill = row['skill']
        imp = row['importance']
        if skill in high_skills:
            priority = 'Critical'
        elif skill in medium_skills:
            priority = 'Growth'
        else:
            priority = 'Development'
        recommendations.append({
            'skill': skill, 
            'priority': priority,
            'importance': imp,
            'reason': f"O*NET importance: {imp:.2f}/5.0 — {priority} org-wide gap"
        })
    
    # Sort: Critical first, then by importance
    priority_order = {'Critical': 0, 'Growth': 1, 'Development': 2}
    recommendations.sort(key=lambda x: (priority_order[x['priority']], -x['importance']))
    return recommendations[:top_n]

# Test V1 for first 3 employees
print("=== V1 Rule-Based Recommendations ===")
test_employees = emp_gap['employee_id'].iloc[:3].tolist()
for emp_id in test_employees:
    recs = v1_recommend(emp_id, skill_gap_detailed, high_severity_skills, medium_severity_skills)
    print(f"\nEmployee {emp_id}:")
    for r in recs:
        print(f"  [{r['priority']}] {r['skill']} (importance={r['importance']:.2f})")


In [ ]:

# ══════════════════════════════════════════════
# V2 — Sentence Transformer + Cosine Similarity
# ══════════════════════════════════════════════
# Strategy: embed each gap skill using sentence-transformers,
# then find the most similar software tools (Workplace Examples)
# from software_skills.csv to recommend concrete tools for each skill.

try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity
    HAS_ST = True
    print("sentence-transformers available — running V2")
except ImportError:
    HAS_ST = False
    print("sentence-transformers not installed — V2 skipped, using V1 results only")


In [ ]:

if HAS_ST:
    # Load model (lightweight)
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # Embed all unique software tools (Workplace Examples)
    sw_tools = sw[['Workplace Example','Element Name','O*NET-SOC Code','Title']].drop_duplicates()
    sw_texts = (sw_tools['Element Name'] + ': ' + sw_tools['Workplace Example']).tolist()
    
    print(f"Encoding {len(sw_texts)} software tools...")
    # Use small batch to avoid memory issues
    sw_embeddings = model.encode(sw_texts, batch_size=64, show_progress_bar=False)
    print(f"Software tool embeddings: {sw_embeddings.shape}")
    
    def v2_recommend_tools(skill_name, sw_tools, sw_embeddings, model, top_n=3):
        skill_emb = model.encode([skill_name])
        sims = cosine_similarity(skill_emb, sw_embeddings)[0]
        top_indices = np.argsort(sims)[::-1][:top_n]
        results = []
        for idx in top_indices:
            tool_row = sw_tools.iloc[idx]
            results.append({
                'tool': tool_row['Workplace Example'],
                'category': tool_row['Element Name'],
                'similarity': round(float(sims[idx]), 4)
            })
        return results
    
    # Test on top 5 gap skills
    print("\n=== V2 Semantic Tool Recommendations ===")
    top_gap_skills = org_gap.nlargest(5, 'total_gap_weight')['skill'].tolist()
    for skill in top_gap_skills:
        tools = v2_recommend_tools(skill, sw_tools, sw_embeddings, model)
        print(f"\nSkill: '{skill}'")
        for t in tools:
            print(f"  → {t['tool']} ({t['category']}) [sim={t['similarity']}]")


In [ ]:

# ── Generate full recommendation table for all employees (V1 + V2) ──
rec_records = []

for _, emp_row in emp_gap.iterrows():
    emp_id = emp_row['employee_id']
    role = emp_row['role']
    v1_recs = v1_recommend(emp_id, skill_gap_detailed, high_severity_skills, medium_severity_skills, top_n=3)
    
    if v1_recs:
        top_skill = v1_recs[0]['skill']
        
        if HAS_ST:
            tools = v2_recommend_tools(top_skill, sw_tools, sw_embeddings, model, top_n=2)
            tool_recs = '; '.join([t['tool'] for t in tools])
        else:
            tool_recs = 'N/A (sentence-transformers not installed)'
        
        rec_records.append({
            'employee_id': emp_id,
            'role': role,
            'top_skill_gap': top_skill,
            'priority': v1_recs[0]['priority'],
            'importance': v1_recs[0]['importance'],
            'all_gaps': '; '.join([r['skill'] for r in v1_recs]),
            'recommended_tools': tool_recs
        })

recs_df = pd.DataFrame(rec_records)
print(f"Recommendations generated for {len(recs_df)} employees")
print(recs_df.head(5).to_string(index=False))


In [ ]:

recs_df.to_csv(f'{PROC}/upskilling_recommendations.csv', index=False)
print("Saved: upskilling_recommendations.csv")


**Recommendation engine complete.** V1 rule-based (Critical/Growth/Development priority). V2 sentence-transformer tool matching (if available).